In [0]:
# ============================================
# GOLD LAYER: Business Aggregations
# ============================================

print("🏆 GOLD LAYER: Business Analytics Tables")
print("=" * 50)

# Load clean silver data
gold_df = spark.table("retail_lakehouse.silver_online_retail")

print(f"📊 Loaded {gold_df.count():,} clean transactions")
print(f"📊 Columns: {', '.join(gold_df.columns)}")

🏆 GOLD LAYER: Business Analytics Tables
📊 Loaded 392,692 clean transactions
📊 Columns: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country, TotalAmount, Year, Month, Day


In [0]:
print("📅 GOLD TABLE 1: Daily Sales Summary")
print("-" * 40)

from pyspark.sql.functions import sum, count, countDistinct, round

daily_sales = gold_df.groupBy("Year", "Month", "Day") \
    .agg(
        round(sum("TotalAmount"), 2).alias("Revenue"),
        sum("Quantity").alias("Total_Items_Sold"),
        countDistinct("InvoiceNo").alias("Total_Orders"),
        countDistinct("CustomerID").alias("Unique_Customers")
    ) \
    .orderBy("Year", "Month", "Day")

# Save
daily_sales.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_lakehouse.gold_daily_sales")

print("✅ gold_daily_sales created")
print(f"📊 {daily_sales.count():,} days of sales data")
daily_sales.show(5)

📅 GOLD TABLE 1: Daily Sales Summary
----------------------------------------
✅ gold_daily_sales created
📊 305 days of sales data
+----+-----+---+--------+----------------+------------+----------------+
|Year|Month|Day| Revenue|Total_Items_Sold|Total_Orders|Unique_Customers|
+----+-----+---+--------+----------------+------------+----------------+
|2010|   12|  1|46192.49|           24114|         121|              95|
|2010|   12|  2|47197.57|           31077|         137|              99|
|2010|   12|  3|23876.63|           11798|          57|              50|
|2010|   12|  5|31361.28|           16241|          87|              75|
|2010|   12|  6|31009.33|           16115|          94|              82|
+----+-----+---+--------+----------------+------------+----------------+
only showing top 5 rows


In [0]:
print("\n📦 GOLD TABLE 2: Product Performance")
print("-" * 40)

# ADD THIS IMPORT
from pyspark.sql.functions import sum, count, countDistinct, round, col

product_performance = gold_df.groupBy("StockCode", "Description") \
    .agg(
        round(sum("TotalAmount"), 2).alias("Total_Revenue"),
        sum("Quantity").alias("Total_Quantity_Sold"),
        countDistinct("InvoiceNo").alias("Order_Count"),
        countDistinct("CustomerID").alias("Unique_Customers")
    ) \
    .orderBy(col("Total_Revenue").desc())

# Save
product_performance.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_lakehouse.gold_product_performance")

print("✅ gold_product_performance created")
print(f"📊 {product_performance.count():,} unique products")
print("\n🏆 Top 5 Products:")
product_performance.show(5, truncate=False)


📦 GOLD TABLE 2: Product Performance
----------------------------------------
✅ gold_product_performance created
📊 3,897 unique products

🏆 Top 5 Products:
+---------+----------------------------------+-------------+-------------------+-----------+----------------+
|StockCode|Description                       |Total_Revenue|Total_Quantity_Sold|Order_Count|Unique_Customers|
+---------+----------------------------------+-------------+-------------------+-----------+----------------+
|23843    |PAPER CRAFT , LITTLE BIRDIE       |168469.6     |80995              |1          |1               |
|22423    |REGENCY CAKESTAND 3 TIER          |142264.75    |12374              |1703       |881             |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|100392.1     |36706              |1971       |856             |
|85099B   |JUMBO BAG RED RETROSPOT           |85040.54     |46078              |1600       |635             |
|23166    |MEDIUM CERAMIC TOP STORAGE JAR    |81416.73     |77916         

In [0]:
print("\n👤 GOLD TABLE 3: Customer Analysis")
print("-" * 40)

# ADD THIS IMPORT
from pyspark.sql.functions import sum, countDistinct, round, col

customer_analysis = gold_df.groupBy("CustomerID") \
    .agg(
        round(sum("TotalAmount"), 2).alias("Total_Spent"),
        countDistinct("InvoiceNo").alias("Total_Orders"),
        sum("Quantity").alias("Total_Items_Bought"),
        countDistinct("StockCode").alias("Unique_Products_Bought"),
        round(sum("TotalAmount") / countDistinct("InvoiceNo"), 2).alias("Avg_Order_Value")
    ) \
    .orderBy(col("Total_Spent").desc())

# Save
customer_analysis.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_lakehouse.gold_customer_analysis")

print("✅ gold_customer_analysis created")
print(f"📊 {customer_analysis.count():,} customers")
print("\n🏆 Top 5 Customers:")
customer_analysis.show(5)


👤 GOLD TABLE 3: Customer Analysis
----------------------------------------
✅ gold_customer_analysis created
📊 4,338 customers

🏆 Top 5 Customers:
+----------+-----------+------------+------------------+----------------------+---------------+
|CustomerID|Total_Spent|Total_Orders|Total_Items_Bought|Unique_Products_Bought|Avg_Order_Value|
+----------+-----------+------------+------------------+----------------------+---------------+
|     14646|  280206.02|          73|            196915|                   700|        3838.44|
|     18102|   259657.3|          60|             64124|                   150|        4327.62|
|     17450|  194390.79|          46|             69973|                   124|        4225.89|
|     16446|   168472.5|           2|             80997|                     3|       84236.25|
|     14911|  143711.17|         201|             80240|                  1787|         714.98|
+----------+-----------+------------+------------------+----------------------+------

In [0]:
print("\n🌍 GOLD TABLE 4: Country Sales")
print("-" * 40)

# ADD THIS IMPORT
from pyspark.sql.functions import sum, countDistinct, round, col

country_sales = gold_df.groupBy("Country") \
    .agg(
        round(sum("TotalAmount"), 2).alias("Total_Revenue"),
        sum("Quantity").alias("Total_Items_Sold"),
        countDistinct("InvoiceNo").alias("Total_Orders"),
        countDistinct("CustomerID").alias("Unique_Customers")
    ) \
    .orderBy(col("Total_Revenue").desc())

# Save
country_sales.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_lakehouse.gold_country_sales")

print("✅ gold_country_sales created")
print(f"📊 {country_sales.count()} countries")
print("\n🌍 Top 10 Countries:")
country_sales.show(10, truncate=False)


🌍 GOLD TABLE 4: Country Sales
----------------------------------------
✅ gold_country_sales created
📊 37 countries

🌍 Top 10 Countries:
+--------------+-------------+----------------+------------+----------------+
|Country       |Total_Revenue|Total_Items_Sold|Total_Orders|Unique_Customers|
+--------------+-------------+----------------+------------+----------------+
|United Kingdom|7285024.64   |4241305         |16646       |3920            |
|Netherlands   |285446.34    |200361          |94          |9               |
|EIRE          |265262.46    |140133          |260         |3               |
|Germany       |228678.4     |119154          |457         |94              |
|France        |208934.31    |111428          |389         |87              |
|Australia     |138453.81    |83891           |57          |9               |
|Spain         |61558.56     |27933           |90          |30              |
|Switzerland   |56443.95     |30082           |51          |21              |
|Belg

In [0]:
print("\n📊 GOLD TABLE 5: Monthly Summary")
print("-" * 40)

# ADD THIS IMPORT
from pyspark.sql.functions import sum, countDistinct, round, col

monthly_summary = gold_df.groupBy("Year", "Month") \
    .agg(
        round(sum("TotalAmount"), 2).alias("Monthly_Revenue"),
        sum("Quantity").alias("Monthly_Items_Sold"),
        countDistinct("InvoiceNo").alias("Monthly_Orders"),
        countDistinct("CustomerID").alias("Monthly_Active_Customers")
    ) \
    .orderBy("Year", "Month")

# Save
monthly_summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_lakehouse.gold_monthly_summary")

print("✅ gold_monthly_summary created")
print(f"📊 {monthly_summary.count()} months of data")
print("\n📈 Monthly Revenue Trend:")
monthly_summary.select("Year", "Month", "Monthly_Revenue").show(15)


📊 GOLD TABLE 5: Monthly Summary
----------------------------------------
✅ gold_monthly_summary created
📊 13 months of data

📈 Monthly Revenue Trend:
+----+-----+---------------+
|Year|Month|Monthly_Revenue|
+----+-----+---------------+
|2010|   12|      570422.73|
|2011|    1|      568101.31|
|2011|    2|      446084.92|
|2011|    3|      594081.76|
|2011|    4|      468374.33|
|2011|    5|      677355.15|
|2011|    6|      660046.05|
|2011|    7|       598962.9|
|2011|    8|      644051.04|
|2011|    9|       950690.2|
|2011|   10|     1035642.45|
|2011|   11|     1156205.61|
|2011|   12|      517190.44|
+----+-----+---------------+



In [0]:
print("\n📋 ALL GOLD TABLES CREATED")
print("=" * 50)

spark.sql("SHOW TABLES IN retail_lakehouse").show(truncate=False)

# Quick row counts
print("\n📊 Table Row Counts:")
for table in ["bronze_online_retail", "silver_online_retail",
              "gold_daily_sales", "gold_product_performance",
              "gold_customer_analysis", "gold_country_sales",
              "gold_monthly_summary"]:
    try:
        cnt = spark.table(f"retail_lakehouse.{table}").count()
        print(f"  {table}: {cnt:,} rows")
    except:
        print(f"  {table}: ❌ not found")


📋 ALL GOLD TABLES CREATED
+----------------+------------------------+-----------+
|database        |tableName               |isTemporary|
+----------------+------------------------+-----------+
|retail_lakehouse|bronze_online_retail    |false      |
|retail_lakehouse|gold_country_sales      |false      |
|retail_lakehouse|gold_customer_analysis  |false      |
|retail_lakehouse|gold_daily_sales        |false      |
|retail_lakehouse|gold_monthly_summary    |false      |
|retail_lakehouse|gold_product_performance|false      |
|retail_lakehouse|silver_online_retail    |false      |
+----------------+------------------------+-----------+


📊 Table Row Counts:
  bronze_online_retail: 541,909 rows
  silver_online_retail: 392,692 rows
  gold_daily_sales: 305 rows
  gold_product_performance: 3,897 rows
  gold_customer_analysis: 4,338 rows
  gold_country_sales: 37 rows
  gold_monthly_summary: 13 rows


In [0]:
print("\n📊 KEY BUSINESS INSIGHTS")
print("=" * 50)

# Top product
top_product = spark.sql("""
    SELECT Description, Total_Revenue 
    FROM retail_lakehouse.gold_product_performance 
    ORDER BY Total_Revenue DESC LIMIT 1
""").collect()[0]
print(f"🏆 Top Product: {top_product[0]}")
print(f"   Revenue: ${top_product[1]:,.2f}")

# Top country
top_country = spark.sql("""
    SELECT Country, Total_Revenue 
    FROM retail_lakehouse.gold_country_sales 
    ORDER BY Total_Revenue DESC LIMIT 1
""").collect()[0]
print(f"\n🌍 Top Country: {top_country[0]}")
print(f"   Revenue: ${top_country[1]:,.2f}")

# Total revenue
total_rev = spark.sql("""
    SELECT ROUND(SUM(Monthly_Revenue), 2) 
    FROM retail_lakehouse.gold_monthly_summary
""").collect()[0][0]
print(f"\n💰 Total Revenue: ${total_rev:,.2f}")

# Customer count
cust_count = spark.table("retail_lakehouse.gold_customer_analysis").count()
print(f"\n👥 Total Customers: {cust_count:,}")

# Avg per customer
print(f"💵 Avg Revenue/Customer: ${total_rev/cust_count:,.2f}")

# Date range
print(f"\n📅 Data Period: 2010-2011 (12 months)")


📊 KEY BUSINESS INSIGHTS
🏆 Top Product: PAPER CRAFT , LITTLE BIRDIE
   Revenue: $168,469.60

🌍 Top Country: United Kingdom
   Revenue: $7,285,024.64

💰 Total Revenue: $8,887,208.89

👥 Total Customers: 4,338
💵 Avg Revenue/Customer: $2,048.69

📅 Data Period: 2010-2011 (12 months)


In [0]:
print("\n" + "=" * 50)
print("🎉 GOLD LAYER COMPLETE!")
print("=" * 50)
print()
print("✅ 5 Gold Tables Created:")
print("   1. gold_daily_sales - Daily revenue & orders")
print("   2. gold_product_performance - Product rankings")
print("   3. gold_customer_analysis - Customer profiles")
print("   4. gold_country_sales - Geographic analysis")
print("   5. gold_monthly_summary - Monthly trends")
print()
print("📁 Next Step: 04_sql_analytics")
print("=" * 50)


🎉 GOLD LAYER COMPLETE!

✅ 5 Gold Tables Created:
   1. gold_daily_sales - Daily revenue & orders
   2. gold_product_performance - Product rankings
   3. gold_customer_analysis - Customer profiles
   4. gold_country_sales - Geographic analysis
   5. gold_monthly_summary - Monthly trends

📁 Next Step: 04_sql_analytics
